# Personal Project Summer 2026
By Evan O'Malley

Context of the project  
- Recreational Project For Resumè Building
- Practice basic dsci and programming skill
- Dataset from Pew Research

Goals of the project  
- Focus on efficient and effective use of library functions
- Focus on making legible work, as if for presentation
- Focus on building a robust and coherent environment for analysis

## Initialization

### Notebook Setup  
Imports files and packages

In [1]:
# Necessary Imports
# Pretty Lean
import pandas as pd, numpy as np
from warnings import simplefilter
simplefilter(action="ignore", category=pd.errors.PerformanceWarning)

rel_path = "Data/Western_Europe_Public_Data_Church_Tax_Added.csv"
codebook_rel_path = "Codebooks/codebook.txt"

### Precursor Structures
Paragraph

In [111]:
class OrderLegend:
    def __init__(self, order: str, legend: dict):
        self.order = order
        self.legend = legend
        
    def ord(self):
        return self.order
    
    def leg(self):
        return self.legend
    
    def values(self):
        return self.legend.values()

    def keys(self):
        return self.legend.keys()

    def items(self):
        return self.legend.items()

    def len(self):
        return len(self.legend)

    def __repr__(self):
        return f"{self.order}, {self.legend}"

    def __str__(self):
        return f"{self.order}, {self.legend}"

    def __getitem__(self, i):
        return self.legend[i]

def Legends_item(item):
    if item in Legends:
        return Legends[item]
    elif item[:-1] in Legends:
        return Legends[item[:-1]]

    raise KeyError

### File Parsing
Makes codebook.txt into usable object

In [3]:
with open(codebook_rel_path) as f:
    codebook = f.read()

def parse_next(terminator):
    assert len(terminator) == 1, "Terminator must be one character"
    global s
    global codebook
    
    passage = ""
    while codebook[s] != terminator:
        passage += codebook[s]
        s += 1
    s += 1
        
    return passage


s = 0
Legends = {}

while s < len(codebook):
    col_legend = {}
    name = parse_next(':')
    order = parse_next('{')
    s += 1
    
    if "[COUNTRY]" in name:
        while codebook[s] != '!':
            idx = int(parse_next(' '))
            val = parse_next('\n')
            col_legend[idx] = val
        s += 1

        Legends[name.replace("[COUNTRY]", "")] = OrderLegend(order, {})
        
        while codebook[s] != '}':
            CTY = codebook[s:s+3]
            s += 5
            
            while codebook[s] != '}':
                idx = int(parse_next(' '))
                val = parse_next('\n')
                col_legend[idx] = val
            s += 2

            col_legend[98] = "Don't know"
            col_legend[99] = "Refused"

            Legends[name.replace("[COUNTRY]", CTY)
            ] = OrderLegend(order, col_legend)
            
        s += 2
            
    else:
        while codebook[s] != '}':
            idx = int(parse_next(' '))
            val = parse_next('\n')
            col_legend[idx] = val
        s += 2

        col_legend[98] = "Don't know"
        col_legend[99] = "Refused"

        Legends[name] = OrderLegend(order, col_legend)

### Parent Restitching
These cells altar the parent DataFrame and cannot be rerun

In [4]:
Parent_DF = pd.read_csv(rel_path, skipinitialspace=True)

In [5]:
# Move QRID column to index
Parent_DF.set_index("QRID", inplace = True)

In [6]:
# Fix respose layout for Q9
q9_cols = Parent_DF.apply(lambda x: x.name[1] == "9")
q9_cols = Parent_DF.loc[:, q9_cols]

def get_Q9(row: pd.Series):
    i = row[row == 1].index
    if len(i) == 0:
        return np.nan
        
    else:
        return int(i[0][3])

Parent_DF.insert(16, "Q9", q9_cols.apply(get_Q9, axis = 1))
Parent_DF.drop(q9_cols, axis = 1, inplace = True)

In [7]:
# Removing QS1... variables becuase they stand for regions and are indecipherable or redacted
# Removing qbornmoverec variable because values are incomprehensible or redacted
QS1_cols = [i for i in Parent_DF.columns if i.lower()[:3] == "qs1"]

Parent_DF.drop(QS1_cols, axis = 1, inplace = True)
Parent_DF.drop("qbornmoverec", axis = 1, inplace = True)

In [8]:
# Knit QIDEOLOGY, QIDEOLOGYa, and QIDEOLOGYb,
# as they are nearly identical
ideology_rec = Parent_DF["QIDEOLOGY"].map(
               lambda x: x + 1 if x < 90 else x)
ideologya_rec = Parent_DF["QIDEOLOGYa"].map(
                lambda x: x + 1 if x< 90 else x)

Parent_DF["QIDEOLOGY"] = Parent_DF["QIDEOLOGYb"].fillna(
                         ideology_rec).fillna(ideologya_rec)

Parent_DF.drop(["QIDEOLOGYa", "QIDEOLOGYb"], axis = 1, inplace = True)

In [9]:
# Knit QDENOM[COUNTRY] columns
# as they use identical enumerations
qdenom = pd.Series([]).reindex(Parent_DF.index)
qdenom_cols = Parent_DF.loc[:, Parent_DF.apply(
              lambda x: x.name[:6].lower() == "qdenom")]

qdenom_cols.apply(lambda x: qdenom.fillna(x, inplace = True))
Parent_DF.insert(26, "QDenom", qdenom)

Parent_DF.drop(qdenom_cols, axis = 1, inplace = True)

In [10]:
# Same goes for QCHDENOM[Country] columns
qchdenom = pd.Series([]).reindex(Parent_DF.index)
qchdenom_cols = Parent_DF.loc[:, Parent_DF.apply(
                lambda x: x.name[:8].lower() == "qchdenom")]

qchdenom_cols.apply(lambda x: qchdenom.fillna(x, inplace = True))
Parent_DF.insert(27, "QChdenom", qchdenom)

Parent_DF.drop(qchdenom_cols, axis = 1, inplace = True)

In [11]:
# Same goes for QCURRELrec and QCURRELDrec columns
qcurrel = Parent_DF["QCURRELrec"].replace([91, 98, 99], np.nan)
Parent_DF["QCURRELrec"] = qcurrel.fillna(Parent_DF["QCURRELDrec"])

Parent_DF.drop("QCURRELDrec", axis = 1, inplace = True)

In [12]:
# Repair column naming scheme
def title_scheme(title: str):
    # Cumulatively adjusts column titles according to scheme described below
    new_title = title

    # Remove instances of "rec", signifying recoded variables

    if new_title[-3:].lower() == "rec":
        new_title = new_title[:-3]

    # Normalize Capitalization Scheme
    # "country" -> "Country"
    # "QCURREL", "qcurrel" -> "QCurrel"
    
    if new_title[0].lower() != 'q':
        new_title = new_title.title()
        
    else:
        new_title = 'Q' + new_title[1:].title()
    
    # Capitalize suffixes signifying country
    # "QDenomaut" -> "QDenomAUT"
    
    if new_title[-3:].upper() in Legends["Country"].values():
        new_title = new_title[:-3] + new_title[-3:].upper()

    if new_title[-4:-1].upper() in Legends["Country"].values():
        new_title = new_title[:-4] + new_title[-4:-1].upper() + new_title[-1]

    # Some questions are divided into cases a, b, c, etcetera
    # Denotation for this will be separated from the main title and uncapitalized
    # "Q4A", "Q4B" -> "Q4_a", Q4_b"
    # These are dicipherable by last 2 characters of the title

    NumCap = new_title[-2].isnumeric() and new_title[-1].isupper()
    CapUncap = new_title[-2].isupper() and new_title[-1].islower() 

    if NumCap or CapUncap:
        new_title = new_title[:-1] + "_" + new_title[-1].lower()

    # Choice adjustments
    if new_title[:4] == "QPty":
        if new_title[4] == 'a':
            new_title = new_title[:4] + "potvot" + new_title[5:]
        elif new_title[4] == 'b':
            new_title = new_title[:4] + "fvr" + new_title[5:]
        else:
            new_title = new_title[:4] + "cls" + new_title[4:]

    rename_key = {"QCitizen1" : "QCitizen",
                  "QBornc" : "QBornmthr",
                  "QBorne" : "QBornfthr",
                  "QChilda" : "QChild",
                  "QHhch" : "QParent"}

    if new_title in rename_key.keys():
        new_title = rename_key[new_title]
    
    return new_title

Parent_DF.rename(title_scheme, axis = 1, inplace = True)

In [26]:
# There are some entires that are undefined by the codebook
def constrain_values(col: pd.Series):
    if Legends_item(col.name).ord() == "Cardinal":
        return col
        
    return col.map(lambda x: x if x in Legends_item(col.name).leg() else np.nan)

Parent_DF = Parent_DF.apply(constrain_values)

### Further Preparations
Functions for table manipulations

In [14]:
# Cardinality function for titles
def cardinality(title: str):
    if title in Legends:
        return Legends[title].ord()
        
    if title[-2] == '_':
        return Legends[title[:-1]].ord()

    raise NotImplementedError(f"Column {title} inappropriately named")

In [15]:
# Function to divide tables into cardinalities
def SepOrder(df: pd.DataFrame, cardinality_arg: str):
    return df.loc[:, df.apply(
           lambda x: cardinality(x.name)) == cardinality_arg]

In [16]:
# Create Generalized list of Qs in order
# This will not be useful later
# strip_countries was though
def strip_countries(title: str):
    # Code primary lifted from title_scheme function
    if title[-3:] in Legends["Country"].values():
        return title[:-3]

    if title[-5:-2] in Legends["Country"].values():
        return title[:-5] + title[-2:]

    return title
        
survey_order = Parent_DF.apply(
               lambda x: strip_countries(x.name)
               ).drop_duplicates().to_numpy()

In [17]:
# Function to disambiguate and ambiguate enumerations
def disambiguate(col: pd.Series):
    if col.dtypes in [int, float]:
        return col.dropna().map(lambda x: Legends[col.name][x])

    else:
        return col

def ambiguate(col: pd.Series):
    if col.dtypes in [int, float]:
        return col

    else:
        reverse_legend = {y:x for x,y in Legends[col.name].items()}
        return col.dropna().map(lambda x: reverse_legend[x])

In [18]:
# Function to divide tables into countries
# Returns a dictionary of the division
def SepCountry(refdf: pd.DataFrame):
    df = refdf.copy()
    country_dfs = {}
    if "Country" not in df:
        df = df.join(Parent_DF["Country"], how = "left")
    df["Country"] = disambiguate(df["Country"])

    for c in df["Country"].dropna().unique():
        country_df = df.copy()
        country_df = country_df.locd[country_df["Country"] == c]
        country_df = country_df.loc[:, country_df.apply(
                     lambda x: any(x.notna()))]

        country_df = country_df.rename(strip_countries, axis = 1)
        country_dfs[c] = country_df.drop("Country", axis = 1)
        
    return country_dfs

In [19]:
# Function to divide tables into belief systems
# Separates Christianity by denomination,
# due to overwhelmingly Christian responses
# Returns a dictionary of the division
def SepReligion(refdf: pd.DataFrame):
    df = refdf.copy()

    if "QCurrel" not in df:
        df = df.join(Parent_DF["QCurrel"], how = "left")
    df["QCurrel"] = disambiguate(df["QCurrel"])
    
    religion_dfs = {}
    for r in df["QCurrel"].dropna().unique():
        if r != "Christian":
            religion_df = df.copy()
            religion_df = religion_df.loc[religion_df["QCurrel"] == r]
            religion_df = religion_df.loc[:, religion_df.apply(
                          lambda x: any(x.notna()))]

            religion_dfs[r] = religion_df.drop("QCurrel", axis = 1)
    
    if "QDenom" not in df:
        df = df.join(Parent_DF["QDenom"], how = "left")
    df["QDenom"] = disambiguate(df["QDenom"])
    
    for d in df["QDenom"].dropna().unique():
        religion_df = df.copy()
        religion_df = religion_df.loc[religion_df["QDenom"] == d]
        religion_df = religion_df.loc[:, religion_df.apply(
                      lambda x: any(x.notna()))]

        religion_dfs[d] = religion_df.drop(["QCurrel", "QDenom"], axis = 1)
            
    return religion_dfs

### Initialization Summary
Coagulate essenstial information  
- Legends object  
- OrderLegend class and methods

Prepare Parent DataFrame  
- Column cropping  
- Rename scheme

Create DataFrame editing tools  
I have not used any of these meaningfully yet  
- SepOrder  
- SepNation
- SepReligion
- Supplementary functions

## Exploratory Analysis

### Statistics Invention
Start inventing some summary statistics

In [20]:
def hist_areas(dfref = pd.DataFrame):
    df = dfref.copy()
    def _responses(x):
        responses = x.value_counts()
        responses[98] = 0
        responses[99] = 0
        responses.drop([98, 99], inplace = True)
        return responses / sum(responses)
        
    return df.apply(_responses)

In [63]:
# Compare response distributions across countries
Ordinal_DF = SepOrder(Parent_DF, "Ordinal")
Ordinal_DF["Country"] = disambiguate(Parent_DF["Country"])
gb_nation = Ordinal_DF.groupby("Country").apply(hist_areas)
gb_nation = gb_nation.loc[:, gb_nation.apply(
            lambda x: x.name == strip_countries(x.name))]

In [64]:
def compare_hist_areas(col: pd.Series, ref_idx = 0):
    frame = Legends_item(col.name).len() - 2

    comp_col = col.groupby(level = 0)
    comp_col = comp_col.apply(lambda x: x.dropna().to_numpy())
    comp_col = comp_col.apply(lambda x: np.pad(x, (0, frame - len(x))))

    ref_arr = col[comp_col.index[ref_idx]].dropna().to_numpy()
    ref_arr = np.pad(ref_arr, (0, frame - len(ref_arr)))
    
    return comp_col.apply(lambda x: sum(np.minimum(x, ref_arr)))

In [299]:
pv_nation = pd.DataFrame()
for r in range(16):
    df_r = gb_nation.apply(compare_hist_areas, ref_idx = r).iloc[r:]
    ref_country = gb_nation.index.get_level_values(0).drop_duplicates()[r]
    df_r.index = pd.MultiIndex.from_arrays(
                   np.stack((np.array([ref_country] * (16-r)), 
                             df_r.index.to_numpy())))
    pv_nation = pd.concat((pv_nation, df_r))

pv_nation.reset_index(inplace = True)
pv_nation.rename({"level_0" : "CTY A",
                  "level_1" : "CTY B"},
                 axis = 1, inplace = True)

In [320]:
pd.pivot_table(pv_nation.melt(id_vars = ["CTY A", "CTY B"]),
               index = "CTY A", columns = "CTY B", values = "value") 

CTY B,AUT,BEG,CHE,DEU,DNK,ESP,FIN,FRA,GBR,IRL,ITA,NLD,NOR,PRT,SLO,SWE
CTY A,,,,,,,,,,,,,,,,
AUT,1.0,0.823987,0.949216,0.951109,0.834105,0.813869,0.811794,0.932922,0.932902,0.930352,0.909259,0.825027,0.806470,0.769744,0.782726,0.768197
BEG,NaN,1.000000,0.809696,0.825472,0.886666,0.868704,0.821039,0.838248,0.818883,0.820014,0.824207,0.883241,0.856361,0.762731,0.798712,0.827853
CHE,NaN,NaN,1.000000,0.941626,0.820472,0.796182,0.796249,0.919950,0.926085,0.923704,0.902360,0.813597,0.792581,0.761317,0.774212,0.761825
DEU,NaN,NaN,NaN,1.000000,0.832841,0.810340,0.801004,0.942696,0.939930,0.916650,0.890009,0.834990,0.808767,0.751335,0.767928,0.773527
DNK,NaN,NaN,NaN,NaN,1.000000,0.833456,0.841097,0.847180,0.828854,0.834856,0.828722,0.881735,0.877328,0.759670,0.804463,0.857750
ESP,NaN,NaN,NaN,NaN,NaN,1.000000,0.829111,0.825182,0.808579,0.829493,0.839349,0.851894,0.833122,0.846342,0.849508,0.780998
FIN,NaN,NaN,NaN,NaN,NaN,NaN,1.000000,0.806884,0.796638,0.828841,0.827907,0.811253,0.827600,0.820294,0.861331,0.769566
FRA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.000000,0.941621,0.926811,0.899279,0.853797,0.828479,0.774257,0.788013,0.799964
GBR,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.000000,0.922725,0.891411,0.835671,0.808372,0.759679,0.765290,0.777249
